# NBM: Neural Basis Model

NBM learns shared neural basis functions for n-ary concept tuples, combines each tuple's basis responses independently, and applies a final linear output layer.


## Model


For tuple $S$ of order $o$ and $K$ learned bases,

$$
b_o(x_S)\in\mathbb R^K,\qquad
h_S(x_S)=a_S^\top b_o(x_S)+c_S,
\qquad
\eta(x)=\beta_0+\sum_S v_Sh_S(x_S).
$$

The default grouped $1\times1$ convolution implements all $a_S$ independently.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import NBMClassifier, NBMLSS, NBMRegressor


model = NBMRegressor(
    nary=[1, 2],
    num_bases=16,
    layer_sizes=[32, 16],
    featurizer="conv1d",
    sparse=False,
    batch_norm=True,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

Use `nary`, `order`, or `interaction_degree` to define tuples. `featurizer='conv1d'` matches upstream; `einsum` is equivalent. Sparse execution is a configuration option, not a separate class.


In [ ]:
einsum_model = NBMRegressor(
    nary=[1, 2], num_bases=16, featurizer="einsum", sparse=False
)
sparse_model = NBMRegressor(
    nary=[1], num_bases=16, sparse=True, nary_ignore_input=0.0
)
if RUN_TRAINING:
    display({key: model.get_params(deep=False)[key] for key in ("nary", "featurizer", "sparse")})
    display(model.predict_components(X_test).terms.keys())


## Task variants and limits

NBM is available as `NBMRegressor`, `NBMClassifier`, and `NBMLSS`. PreTab emits one scalar encoded column per NBM concept.
